평가 기준은 [평가 기준 문서](../docs/evaluation_criteria.md)를 참고하세요.

## 평가 개요

- 세 실험의 저장된 가중치를 순서대로 불러와 심비대·폐부종·흉수 소견 탐지 성능을 평가합니다.
- 현재 평가 데이터는 `data/processed/pilot/pilot_test.csv`이며, 평가 이미지 수는 해당 CSV를 기준으로 결정됩니다.
- 학습과 동일한 전처리를 적용하고, 전체 테스트 이미지의 정답·예측 확률·예측값을 수집합니다.

## 비교 실험

| 실험 | 모델 | 불확실 라벨 처리 | 비교 목적 |
| --- | --- | --- | --- |
| ① 기준 | DenseNet121 | U-multiclass | 기준 실험 |
| ② 모델 비교 | ResNet50 | U-multiclass | ①과 비교하여 모델 구조의 효과 확인 |
| ③ 라벨 처리 비교 | DenseNet121 | U-ignore | ①과 비교하여 불확실 라벨 처리 방식의 효과 확인 |

모델 종류 비교는 ①과 ②, 라벨 처리 방식 비교는 ①과 ③을 대상으로 합니다.

## 공통 평가 기준

- 세 실험은 동일한 테스트 데이터와 전처리 설정으로 비교합니다.
- 소견별 이진 평가에서는 해당 소견의 정답이 음성(0)·양성(1)인 항목만 사용합니다.
- 단일·복합 소견별 평가에서는 세 소견의 정답이 모두 음성·양성인 이미지만 사용합니다.
- 공통 양성 점수는 U-multiclass의 경우 `양성 확률 / (음성 확률 + 양성 확률)`, U-ignore의 경우 음성·양성 2클래스 softmax의 양성 확률을 사용합니다.
- 공통 이진 판정 기준값의 기본값은 `0.5`이며, 양성 점수가 기준값 이상이면 양성으로 판정합니다. 기준값을 조정할 경우 검증 데이터에서 결정한 값을 적용합니다.
- AUROC·AP·AUPRC(사다리꼴)는 공통 양성 점수로, Precision·Recall·F1·민감도·특이도는 공통 이진 예측으로 계산합니다.
- U-multiclass의 3클래스 분류 보고서·혼동행렬과 정규화 전 양성 확률 AUROC는 부가 분석으로 제공합니다.

## 결과 저장 및 해석

- 세 실험을 자동으로 순차 평가하고, 예측 결과·평가 지표·시각화·오분류 사례·평가 설정을 저장합니다.
- 저장 위치: `evaluation/results/<실험 이름>/<학습 실행 폴더>/`
- 현재 노트북의 pilot 평가 결과는 최종 학습 모델의 평가 결과와 구분합니다. 최종 평가는 독립된 테스트 데이터에서 수행합니다.


## 자동 실행 방법

1. 아래 설정 셀에서 프로젝트·이미지 경로와 세 체크포인트 경로를 확인합니다.
2. GPU 서버의 Python 커널을 선택하고 **전체 실행(Run All)** 합니다.
3. 세 모델을 순서대로 평가하고 실험별 CSV·JSON·그래프를 저장합니다.
4. 마지막에 9행의 소견별 비교표와 단일·복합 그룹 비교표를 저장합니다.

각 모델의 평가가 끝나면 GPU 메모리를 정리합니다. 오류가 발생하면 해당 실험에서 실행을 중단하며, 앞서 저장한 실험 결과는 유지됩니다. 같은 설정으로 다시 실행하면 같은 결과 폴더의 파일을 갱신합니다.


### 체크포인트 입력 설정

U-multiclass는 `input`을 사용합니다. 확인한 U-ignore 형식은 `config.image_size`와 공통 전처리 코드에서 입력 설정을 복원합니다. 설정의 출처도 결과 JSON에 기록합니다. 평가 정답은 모든 실험에서 0/1/2로 보존합니다.


### U-ignore 출력 구조

확인한 학습 코드 `DenseNetUIgnore`와 동일하게 출력층을 6개로 구성하고 `(N, 3, 2)`로 변환합니다. 클래스 순서는 음성 → 양성이며, 소견별 2클래스 softmax의 양성 확률을 공통 점수로 사용합니다. `y_prob`에는 두 클래스의 확률을 모두 저장합니다. `y_pred`는 모델의 argmax 예측이고, 공통 평가는 기준값을 적용한 `y_pred_binary`를 사용합니다.


## Bootstrap 95% 신뢰구간
- 반복 1,000회, 난수 시드 42, percentile 방식입니다.
- 공통 AUROC·AUPRC·AP·양성 Precision·양성 F1·민감도·특이도에 CI를 계산합니다. 모든 지표는 소견별 불확실 정답을 제외합니다.
- subject_id가 있으면 환자 단위로 복원추출하며, 선택한 환자의 모든 영상을 함께 포함합니다. 없으면 이미지 단위로 계산하고 안내합니다.
- 유효 반복 횟수를 지표별로 기록합니다. AUROC·AUPRC·AP는 음성·양성이 모두 있는 표본만 사용합니다.
- 실험별 confidence_intervals.csv와 최종 experiment_comparison.csv에 CI를 저장합니다.
- CI는 고정된 모델의 테스트 표본 변동을 반영하며 재학습 변동이나 모델 간 차이의 CI는 아닙니다.


## 1. 공통 설정 및 세 실험 경로


In [ ]:
from pathlib import Path
import gc
import json
import sys
import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT_DIR = Path('/home/team30103/chest-xray-multilabel')
IMAGE_ROOT = Path('/home/team30103/physionet_data')
CSV_DIR = PROJECT_DIR / 'data' / 'processed' / 'pilot'
RESULTS_ROOT = PROJECT_DIR / 'evaluation' / 'results'
THRESHOLD = 0.5
BATCH_SIZE = 4
SHOW_PLOTS = True  # False여도 그래프 파일은 저장

EXPERIMENTS = [
    {
        'model_name': 'densenet121',
        'uncertainty_strategy': 'u_multiclass',
        'model_path': PROJECT_DIR / 'runs/pilot/train_20261015_095921_874554/best_model.pth',
    },
    {
        'model_name': 'resnet50',
        'uncertainty_strategy': 'u_multiclass',
        'model_path': PROJECT_DIR / 'runs/pilot_resnet50/train_20261016_141814_378177/best_model.pth',
    },
    {
        'model_name': 'densenet121',
        'uncertainty_strategy': 'u_ignore',
        'model_path': PROJECT_DIR / 'runs/uignore/train_20261016_150548_221830/best_model.pth',
    },
]

sys.path.insert(0, str(PROJECT_DIR))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('사용 장치:', device)

# Bootstrap percentile 95% CI
N_BOOTSTRAP = 1000
CI_LEVEL = 0.95
CI_SEED = 42
CI_METRICS = ['공통 AUROC', '공통 AUPRC (사다리꼴)', 'AP', '양성 Precision', '양성 F1', 'Sensitivity', 'Specificity']


## 2. 공통 라벨 및 모델 구조


In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
)

# 소견 순서
FINDINGS = ["Cardiomegaly", "Edema", "Pleural Effusion"]
FINDING_NAMES = ["심비대", "폐부종", "흉수"]

# 클래스 순서
CLASS_NAMES = ["음성", "양성", "불확실"]
CLASS_LABELS = [0, 1, 2]


In [ ]:
import torch
from torch import nn
from torchvision.models import densenet121, resnet50




# 선택한 모델·라벨 처리 방식에 맞는 구조
class CXRModel(nn.Module):
    def __init__(self, model_name, uncertainty_strategy):
        super().__init__()

        if uncertainty_strategy not in ("u_multiclass", "u_ignore"):
            raise ValueError(
                f"지원하지 않는 라벨 처리 방식: {uncertainty_strategy}"
            )

        self.uncertainty_strategy = uncertainty_strategy

        # U-multiclass: 3개 소견 × 3개 클래스
        # U-ignore: 3개 소견 × 음성·양성 2개 클래스
        output_size = (
            9 if uncertainty_strategy == "u_multiclass" else 6
        )

        if model_name == "densenet121":
            self.backbone = densenet121(weights=None)
            self.backbone.classifier = nn.Linear(
                self.backbone.classifier.in_features,
                output_size
            )

        elif model_name == "resnet50":
            self.backbone = resnet50(weights=None)
            self.backbone.fc = nn.Linear(
                self.backbone.fc.in_features,
                output_size
            )

        else:
            raise ValueError(
                f"지원하지 않는 모델: {model_name}"
            )

    def forward(self, images):
        logits = self.backbone(images)

        if self.uncertainty_strategy == "u_multiclass":
            return logits.reshape(-1, 3, 3)

        return logits.reshape(-1, 3, 2)


## 3. 실행 전 확인


In [ ]:

def validate_checkpoint_output(checkpoint, model_name, uncertainty_strategy):
    expected_names = (['negative', 'positive', 'uncertain']
                      if uncertainty_strategy == 'u_multiclass'
                      else ['negative', 'positive'])
    if list(checkpoint.get('class_names', [])) != expected_names:
        raise ValueError('체크포인트의 클래스 순서가 다릅니다.')
    if list(checkpoint.get('target_labels', [])) != FINDINGS:
        raise ValueError('체크포인트의 소견 순서가 다릅니다.')
    if checkpoint.get('label_strategy', uncertainty_strategy) != uncertainty_strategy:
        raise ValueError('체크포인트의 라벨 처리 방식이 다릅니다.')
    if model_name not in ('densenet121', 'resnet50'):
        raise ValueError(f'지원하지 않는 모델: {model_name}')
    head = 'classifier' if model_name == 'densenet121' else 'fc'
    input_features = 1024 if model_name == 'densenet121' else 2048
    output_features = 3 * len(expected_names)
    state = checkpoint['model_state_dict']
    for suffix, shape in [('weight', (output_features, input_features)),
                          ('bias', (output_features,))]:
        key = f'backbone.{head}.{suffix}'
        if key not in state or tuple(state[key].shape) != shape:
            actual = tuple(state[key].shape) if key in state else 'missing'
            raise ValueError(f'모델 출력층 불일치: {key}, 기대 {shape}, 실제 {actual}')


def resolve_input_config(checkpoint, uncertainty_strategy):
    """학습 체크포인트와 확인한 공통 전처리 코드에서 입력 설정을 복원합니다."""
    from cxr_preprocessing import MEAN, STD

    saved = checkpoint.get('input')
    if saved is not None:
        if not isinstance(saved, dict):
            raise ValueError('체크포인트 input은 딕셔너리여야 합니다.')
        height, width = saved.get('height'), saved.get('width')
        source = 'checkpoint.input'
    else:
        # 확인한 U-ignore 학습 코드는 공통 이미지 전처리를 그대로 재사용합니다.
        preprocessing = checkpoint.get('preprocessing', '')
        if (uncertainty_strategy != 'u_ignore'
                or not isinstance(preprocessing, str)
                or preprocessing.strip() != 'cxr_preprocessing.py / cxr_preprocessing_uignore.py'):
            raise ValueError('input이 없고 확인한 U-ignore 전처리 형식도 아닙니다.')
        height = checkpoint.get('config', {}).get('image_size')
        width = height
        source = 'checkpoint.config.image_size + reviewed shared preprocessing'

    if (not isinstance(height, int) or isinstance(height, bool) or height <= 0
            or not isinstance(width, int) or width != height):
        raise ValueError('유효한 정사각형 입력 크기를 확인할 수 없습니다.')

    effective = {
        'channels': 3, 'height': height, 'width': width,
        'scale': 'uint8 / 255', 'mean': list(MEAN), 'std': list(STD),
        'geometry': 'preserve aspect ratio, pad square, resize',
    }
    if saved is not None:
        for key, expected in effective.items():
            actual = saved.get(key)
            if key in ('mean', 'std'):
                if actual is None or not np.allclose(actual, expected, rtol=0, atol=1e-7):
                    raise ValueError(f'공통 전처리와 체크포인트 설정이 다릅니다: {key}')
            elif actual != expected:
                raise ValueError(f'공통 전처리와 체크포인트 설정이 다릅니다: {key}')
    return effective, source

def preflight_experiments():
    required = [CSV_DIR / 'pilot_test.csv', PROJECT_DIR / 'cxr_preprocessing.py']
    required += [Path(exp['model_path']) for exp in EXPERIMENTS]
    for path in required:
        if not path.is_file():
            raise FileNotFoundError(f'필수 파일이 없습니다: {path}')
    if not IMAGE_ROOT.is_dir():
        raise FileNotFoundError(f'이미지 폴더가 없습니다: {IMAGE_ROOT}')
    if not 0 <= THRESHOLD <= 1:
        raise ValueError('THRESHOLD는 0~1이어야 합니다.')
    names = [f"{e['model_name']}_{e['uncertainty_strategy']}" for e in EXPERIMENTS]
    if len(names) != len(set(names)):
        raise ValueError('중복된 실험 이름이 있습니다.')
    if not EXPERIMENTS:
        raise ValueError('평가할 실험이 없습니다.')
    # 전체 추론 전에 체크포인트별 입력 설정과 소견 순서 확인
    reference_input = None
    for exp in EXPERIMENTS:
        ckpt = torch.load(exp['model_path'], map_location='cpu', weights_only=True)
        validate_checkpoint_output(ckpt, exp['model_name'], exp['uncertainty_strategy'])
        if ckpt.get('smoke_test', False):
            raise ValueError(f"임시 데이터 학습 모델입니다: {exp['model_path']}")
        if list(ckpt['target_labels']) != FINDINGS:
            raise ValueError('체크포인트의 소견 순서가 다릅니다.')
        input_config, _ = resolve_input_config(ckpt, exp['uncertainty_strategy'])
        if reference_input is None:
            reference_input = input_config
        elif input_config != reference_input:
            raise ValueError('모델별 input 설정이 다릅니다. 학습 전처리를 먼저 확인하세요.')
        del ckpt
    print('세 실험의 필수 파일 및 입력 설정 확인 완료')


def calculate_binary_metrics(truth, score, predicted):
    from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, auc
    mask = np.isin(truth, [0, 1])
    truth, score, predicted = truth[mask], score[mask], predicted[mask]
    result = {name: np.nan for name in CI_METRICS}
    if not len(truth):
        return result
    tp = int(((truth == 1) & (predicted == 1)).sum())
    fn = int(((truth == 1) & (predicted == 0)).sum())
    tn = int(((truth == 0) & (predicted == 0)).sum())
    fp = int(((truth == 0) & (predicted == 1)).sum())
    # Precision/F1 follow classification_report(zero_division=0).
    result['양성 Precision'] = tp / (tp + fp) if tp + fp else 0.0
    result['양성 F1'] = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
    result['Sensitivity'] = tp / (tp + fn) if tp + fn else np.nan
    result['Specificity'] = tn / (tn + fp) if tn + fp else np.nan
    if len(np.unique(truth)) == 2:
        result['공통 AUROC'] = roc_auc_score(truth, score)
        result['AP'] = average_precision_score(truth, score)
        precision, recall, _ = precision_recall_curve(truth, score)
        result['공통 AUPRC (사다리꼴)'] = auc(recall, precision)
    return result


def bootstrap_confidence_intervals(test_df, y_true, y_score, y_pred_binary):
    if not (len(test_df) == len(y_true) == len(y_score) == len(y_pred_binary)):
        raise ValueError('테스트 CSV와 예측 결과의 행 수가 다릅니다.')
    if N_BOOTSTRAP < 1 or not 0 < CI_LEVEL < 1:
        raise ValueError('Bootstrap 반복 횟수와 신뢰수준을 확인하세요.')
    rng = np.random.default_rng(CI_SEED)
    n = len(y_true)
    if not n:
        raise ValueError('평가할 이미지가 없습니다.')
    if 'subject_id' in test_df.columns:
        if test_df['subject_id'].isna().any():
            raise ValueError('subject_id에 결측값이 있습니다.')
        ids = test_df['subject_id'].astype(str).to_numpy()
        if any(not x.strip() for x in ids):
            raise ValueError('subject_id에 빈 값이 있습니다.')
        codes, patients = pd.factorize(ids, sort=False)
        units = [np.flatnonzero(codes == k) for k in range(len(patients))]
        sampling_unit = '환자'
    else:
        units = None
        sampling_unit = '이미지'
        print('subject_id가 없어 이미지 단위 CI를 계산합니다. 동일 환자 영상이 여러 장이면 subject_id를 추가하세요.')
    samples = [{name: [] for name in CI_METRICS} for _ in FINDING_NAMES]
    points = [calculate_binary_metrics(y_true[:, i], y_score[:, i], y_pred_binary[:, i]) for i in range(3)]
    for repeat in range(N_BOOTSTRAP):
        if units is None:
            indices = rng.integers(0, n, size=n)
        else:
            chosen = rng.integers(0, len(units), size=len(units))
            indices = np.concatenate([units[k] for k in chosen])
        # Share sampled rows across findings; reset the seed for each experiment.
        for i in range(3):
            metrics = calculate_binary_metrics(y_true[indices, i], y_score[indices, i], y_pred_binary[indices, i])
            for name, value in metrics.items():
                if np.isfinite(value):
                    samples[i][name].append(value)
        if (repeat + 1) % 100 == 0:
            print(f'Bootstrap 진행: {repeat + 1}/{N_BOOTSTRAP}')
    alpha = (1 - CI_LEVEL) / 2
    rows = []
    for i, finding in enumerate(FINDING_NAMES):
        for name in CI_METRICS:
            values = samples[i][name]
            lower, upper = np.quantile(values, [alpha, 1 - alpha]) if values else (np.nan, np.nan)
            rows.append({'소견': finding, '지표': name, '평가 이미지 수': int(np.isin(y_true[:, i], [0, 1]).sum()),
                         '점추정치': points[i][name], '95% CI 하한': lower, '95% CI 상한': upper,
                         '유효 반복 횟수': len(values)})
    config = {'method': 'bootstrap percentile', 'confidence_level': CI_LEVEL, 'n_bootstrap': N_BOOTSTRAP,
              'seed': CI_SEED, 'sampling_unit': sampling_unit, 'sampling_unit_count': len(units) if units is not None else n,
              'finding_rule': '각 소견의 음성·양성 정답만 평가; 불확실 정답 제외',
              'scope': '고정된 학습 모델의 테스트 표본 변동; 재학습 변동 미포함'}
    return pd.DataFrame(rows), config


## 4. 단일 실험 평가 및 순차 실행 함수

공통 이진 평가와 U-multiclass 부가 분석 기준을 유지합니다. 모든 결과는 실험별 폴더에 저장합니다.


In [ ]:
def evaluate_experiment(experiment):
    MODEL_NAME = experiment['model_name']
    UNCERTAINTY_STRATEGY = experiment['uncertainty_strategy']
    MODEL_PATH = Path(experiment['model_path'])
    EXPERIMENT_NAME = f'{MODEL_NAME}_{UNCERTAINTY_STRATEGY}'
    model = None
    print(f'\n===== {EXPERIMENT_NAME} 평가 시작 =====')
    try:
        checkpoint = torch.load(MODEL_PATH, map_location='cpu', weights_only=True)
        input_config, input_config_source = resolve_input_config(checkpoint, UNCERTAINTY_STRATEGY)
        validate_checkpoint_output(checkpoint, MODEL_NAME, UNCERTAINTY_STRATEGY)
        assert checkpoint['target_labels'] == ['Cardiomegaly', 'Edema', 'Pleural Effusion'], '체크포인트의 소견 순서가 평가 코드와 다릅니다.'
        if UNCERTAINTY_STRATEGY == 'u_multiclass':
            assert checkpoint['class_names'] == ['negative', 'positive', 'uncertain'], '체크포인트의 클래스 순서가 평가 코드와 다릅니다.'
        if checkpoint.get('smoke_test', False):
            raise ValueError('임시 데이터로 학습한 demo 모델입니다.')
        model = CXRModel(model_name=MODEL_NAME, uncertainty_strategy=UNCERTAINTY_STRATEGY)
        model.load_state_dict(checkpoint['model_state_dict'], strict=True)
        model.to(device)
        model.eval()
        print('실험 이름:', EXPERIMENT_NAME)
        print('사용 장치:', device)
        print('저장 당시 epoch:', checkpoint['epoch'])
        print('학습된 모델 불러오기 완료')
        import sys
        import pandas as pd
        from torch.utils.data import DataLoader
        sys.path.insert(0, str(PROJECT_DIR))
        from cxr_preprocessing import CXRDataset
        test_df = pd.read_csv(CSV_DIR / 'pilot_test.csv', dtype={'dicom_id': 'string'})
        test_df['image_path'] = test_df['dicom_id'].map(lambda image_id: str(IMAGE_ROOT / f'{image_id}.jpg'))
        missing = test_df['image_path'].map(lambda path: not Path(path).is_file())
        if missing.any():
            raise FileNotFoundError(f"test 이미지 {missing.sum()}장 누락. 첫 누락 경로: {test_df.loc[missing, 'image_path'].iloc[0]}")
        if test_df.empty:
            raise ValueError('테스트 CSV에 이미지가 없습니다.')
        if test_df['dicom_id'].isna().any() or test_df['dicom_id'].duplicated().any():
            raise ValueError('테스트 CSV의 dicom_id 결측 또는 중복을 확인하세요.')
        test_dataset = CXRDataset(test_df, image_size=input_config['height'])
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)
        print('test 이미지 수:', len(test_dataset))
        print('test 배치 수:', len(test_loader))
        print('test DataLoader 준비 완료')
        true_batches = []
        prob_batches = []
        score_batches = []
        model.eval()
        with torch.inference_mode():
            for batch_idx, (images, labels) in enumerate(test_loader, start=1):
                images = images.to(device)
                logits = model(images).float()
                if UNCERTAINTY_STRATEGY == 'u_multiclass':
                    probs = logits.softmax(dim=-1)
                    scores = logits[..., :2].softmax(dim=-1)[..., 1]
                else:
                    probs = logits.softmax(dim=-1)
                    scores = probs[..., 1]
                true_batches.append(labels.cpu())
                prob_batches.append(probs.cpu())
                score_batches.append(scores.cpu())
                print(f'예측 진행: {batch_idx}/{len(test_loader)} 배치')
        y_true = torch.cat(true_batches).numpy()
        y_prob = torch.cat(prob_batches).numpy()
        y_score = torch.cat(score_batches).numpy()
        y_pred_binary = (y_score >= THRESHOLD).astype(np.int64)
        if UNCERTAINTY_STRATEGY == 'u_multiclass':
            y_pred = y_prob.argmax(axis=-1)
        else:
            y_pred = y_prob.argmax(axis=-1)
        print('정답 형태:', y_true.shape)
        print('모델 확률 형태:', y_prob.shape)
        print('모델 예측 형태:', y_pred.shape)
        print('공통 양성 점수 형태:', y_score.shape)
        print('공통 이진 예측 형태:', y_pred_binary.shape)
        n_images = len(y_true)
        assert y_true.shape == (n_images, 3)
        assert y_pred.shape == (n_images, 3)
        assert y_score.shape == (n_images, 3)
        assert y_pred_binary.shape == (n_images, 3)
        assert np.isin(y_true, [0, 1, 2]).all()
        assert np.isfinite(y_prob).all()
        assert ((y_prob >= 0) & (y_prob <= 1)).all()
        assert np.isfinite(y_score).all()
        assert ((y_score >= 0) & (y_score <= 1)).all()
        assert 0 <= THRESHOLD <= 1
        assert np.isin(y_pred_binary, [0, 1]).all()
        assert np.array_equal(y_pred_binary, (y_score >= THRESHOLD).astype(np.int64))
        if UNCERTAINTY_STRATEGY == 'u_multiclass':
            assert y_prob.shape == (n_images, 3, 3)
            assert np.allclose(y_prob.sum(axis=-1), 1.0, atol=1e-05)
            assert np.array_equal(y_pred, y_prob.argmax(axis=-1))
        elif UNCERTAINTY_STRATEGY == 'u_ignore':
            assert y_prob.shape == (n_images, 3, 2)
            assert np.allclose(y_prob.sum(axis=-1), 1.0, atol=1e-5)
            assert np.allclose(y_score, y_prob[..., 1])
            assert np.array_equal(y_pred, y_prob.argmax(axis=-1))
        else:
            raise ValueError(f'지원하지 않는 라벨 처리 방식: {UNCERTAINTY_STRATEGY}')
        print('평가 데이터 검증 완료')
        print('실험 이름:', EXPERIMENT_NAME)
        print('평가 이미지 수:', n_images)
        print('공통 이진 판정 기준:', THRESHOLD)
        BINARY_CLASS_NAMES = ['음성', '양성']
        BINARY_CLASS_LABELS = [0, 1]
        for i, finding in enumerate(FINDING_NAMES):
            mask = np.isin(y_true[:, i], [0, 1])
            true_binary = y_true[mask, i]
            pred_binary = y_pred_binary[mask, i]
            print(f'\n===== {finding}: 공통 이진 평가 =====')
            print('평가 이미지 수:', int(mask.sum()))
            print('불확실 정답 제외 수:', int((~mask).sum()))
            if len(true_binary) == 0:
                print('평가 가능한 음성·양성 정답이 없습니다.')
                continue
            print(classification_report(true_binary, pred_binary, labels=BINARY_CLASS_LABELS, target_names=BINARY_CLASS_NAMES, zero_division=0))
        if UNCERTAINTY_STRATEGY == 'u_multiclass':
            for i, finding in enumerate(FINDING_NAMES):
                print(f'\n===== {finding}: 3클래스 부가 분석 =====')
                print(classification_report(y_true[:, i], y_pred[:, i], labels=CLASS_LABELS, target_names=CLASS_NAMES, zero_division=0))
        for i, finding in enumerate(FINDING_NAMES):
            mask = np.isin(y_true[:, i], [0, 1])
            print(f'\n===== {finding}: 공통 이진 혼동행렬 =====')
            if not mask.any():
                print('평가 가능한 음성·양성 정답이 없습니다.')
                continue
            cm = confusion_matrix(y_true[mask, i], y_pred_binary[mask, i], labels=BINARY_CLASS_LABELS)
            cm_df = pd.DataFrame(cm, index=[f'실제 {name}' for name in BINARY_CLASS_NAMES], columns=[f'예측 {name}' for name in BINARY_CLASS_NAMES])
            display(cm_df)
        if UNCERTAINTY_STRATEGY == 'u_multiclass':
            for i, finding in enumerate(FINDING_NAMES):
                cm = confusion_matrix(y_true[:, i], y_pred[:, i], labels=CLASS_LABELS)
                cm_df = pd.DataFrame(cm, index=[f'실제 {name}' for name in CLASS_NAMES], columns=[f'예측 {name}' for name in CLASS_NAMES])
                print(f'\n===== {finding}: 3클래스 부가 분석 =====')
                display(cm_df)
        auroc_results = []
        for i, finding in enumerate(FINDING_NAMES):
            mask = np.isin(y_true[:, i], [0, 1])
            true_binary = y_true[mask, i]
            positive_score = y_score[mask, i]
            common_auroc = np.nan
            raw_auroc = np.nan
            if len(np.unique(true_binary)) == 2:
                common_auroc = roc_auc_score(true_binary, positive_score)
                if UNCERTAINTY_STRATEGY == 'u_multiclass':
                    raw_auroc = roc_auc_score(true_binary, y_prob[mask, i, 1])
            result = {'소견': finding, '평가 이미지 수': int(mask.sum()), '음성 수': int((true_binary == 0).sum()), '양성 수': int((true_binary == 1).sum()), '공통 AUROC': common_auroc}
            if UNCERTAINTY_STRATEGY == 'u_multiclass':
                result['3클래스 양성 확률 AUROC (부가)'] = raw_auroc
            auroc_results.append(result)
        auroc_df = pd.DataFrame(auroc_results)
        display(auroc_df)
        from sklearn.metrics import precision_recall_curve, auc
        auprc_results = []
        for i, finding in enumerate(FINDING_NAMES):
            mask = np.isin(y_true[:, i], [0, 1])
            true_binary = y_true[mask, i]
            positive_score = y_score[mask, i]
            auprc = np.nan
            if len(np.unique(true_binary)) == 2:
                precision, recall, _ = precision_recall_curve(true_binary, positive_score)
                auprc = auc(recall, precision)
            auprc_results.append({'소견': finding, '평가 이미지 수': int(mask.sum()), '양성 비율': float((true_binary == 1).mean()) if len(true_binary) > 0 else np.nan, '공통 AUPRC (사다리꼴)': auprc})
        auprc_df = pd.DataFrame(auprc_results)
        display(auprc_df)
        from sklearn.metrics import average_precision_score
        additional_results = []
        for i, finding in enumerate(FINDING_NAMES):
            mask = np.isin(y_true[:, i], [0, 1])
            true = y_true[mask, i]
            pred = y_pred_binary[mask, i]
            positive_score = y_score[mask, i]
            tp = np.sum((true == 1) & (pred == 1))
            fn = np.sum((true == 1) & (pred == 0))
            tn = np.sum((true == 0) & (pred == 0))
            fp = np.sum((true == 0) & (pred == 1))
            sensitivity = tp / (tp + fn) if tp + fn > 0 else np.nan
            specificity = tn / (tn + fp) if tn + fp > 0 else np.nan
            ap = np.nan
            if len(np.unique(true)) == 2:
                ap = average_precision_score(true, positive_score)
            additional_results.append({'소견': finding, '평가 이미지 수': int(mask.sum()), 'Sensitivity': sensitivity, 'Specificity': specificity, 'AP': ap})
        additional_df = pd.DataFrame(additional_results)
        display(additional_df)
        ci_df, ci_config = bootstrap_confidence_intervals(test_df, y_true, y_score, y_pred_binary)
        display(ci_df.round(4))
        certain_mask = np.isin(y_true, [0, 1]).all(axis=1)
        positive_count = (y_true == 1).sum(axis=1)
        groups = {'양성 소견 0개': certain_mask & (positive_count == 0), '단일 소견 (양성 1개)': certain_mask & (positive_count == 1), '복합 소견 (양성 2~3개)': certain_mask & (positive_count >= 2)}
        group_results = []
        for group_name, mask in groups.items():
            n = int(mask.sum())
            if n > 0:
                matches = y_true[mask] == y_pred_binary[mask]
                exact_match = matches.all(axis=1).mean()
                finding_accuracy = matches.mean()
            else:
                exact_match = np.nan
                finding_accuracy = np.nan
            group_results.append({'그룹': group_name, '이미지 수': n, '세 소견 모두 정답 비율': exact_match, '소견별 판단 정확도': finding_accuracy})
        group_df = pd.DataFrame(group_results)
        display(group_df)
        import matplotlib.pyplot as plt
        import matplotlib.font_manager as fm
        font_path = PROJECT_DIR / 'evaluation' / 'MALGUN.TTF'
        if font_path.is_file():
            fm.fontManager.addfont(str(font_path))
            plt.rcParams['font.family'] = fm.FontProperties(fname=str(font_path)).get_name()
        else:
            print('MALGUN.TTF가 없어 기본 폰트를 사용합니다. 한글이 깨질 수 있습니다.')
        plt.rcParams['axes.unicode_minus'] = False
        SAVE_DIR = RESULTS_ROOT / EXPERIMENT_NAME / MODEL_PATH.parent.name
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        import matplotlib.pyplot as plt
        import seaborn as sns
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        for i, finding in enumerate(FINDING_NAMES):
            mask = np.isin(y_true[:, i], [0, 1])
            axes[i].set_title(finding)
            if not mask.any():
                axes[i].text(0.5, 0.5, '평가 가능한 정답 없음', ha='center', va='center', transform=axes[i].transAxes)
                axes[i].set_axis_off()
                continue
            cm = confusion_matrix(y_true[mask, i], y_pred_binary[mask, i], labels=BINARY_CLASS_LABELS)
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', vmin=0, vmax=len(y_true), xticklabels=BINARY_CLASS_NAMES, yticklabels=BINARY_CLASS_NAMES, ax=axes[i])
            axes[i].set_xlabel('예측')
            axes[i].set_ylabel('실제 정답')
        fig.suptitle(f'{EXPERIMENT_NAME} | 공통 이진 혼동행렬')
        fig.tight_layout(rect=[0, 0, 1, 0.92])
        fig.savefig(SAVE_DIR / 'confusion_matrix_binary.png', dpi=300, bbox_inches='tight')
        if SHOW_PLOTS:
            plt.show()
        plt.close(fig)
        if UNCERTAINTY_STRATEGY == 'u_multiclass':
            fig, axes = plt.subplots(1, 3, figsize=(15, 4))
            for i, finding in enumerate(FINDING_NAMES):
                cm = confusion_matrix(y_true[:, i], y_pred[:, i], labels=CLASS_LABELS)
                sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', vmin=0, vmax=len(y_true), xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[i])
                axes[i].set_title(finding)
                axes[i].set_xlabel('예측')
                axes[i].set_ylabel('실제 정답')
            fig.suptitle(f'{EXPERIMENT_NAME} | 3클래스 부가 분석')
            fig.tight_layout(rect=[0, 0, 1, 0.92])
            fig.savefig(SAVE_DIR / 'confusion_matrix_multiclass.png', dpi=300, bbox_inches='tight')
            if SHOW_PLOTS:
                plt.show()
            plt.close(fig)
        ax = group_df.plot(x='그룹', y=['세 소견 모두 정답 비율', '소견별 판단 정확도'], kind='bar', figsize=(10, 5), rot=0, color=['steelblue', 'orange'])
        ax.set_title('양성 소견 개수에 따른 성능 비교')
        ax.set_xlabel('그룹')
        ax.set_ylabel('정확도')
        ax.set_ylim(0, 1.1)
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
        plt.tight_layout()
        plt.savefig(SAVE_DIR / 'finding_groups.png', dpi=300, bbox_inches='tight')
        if SHOW_PLOTS:
            plt.show()
        assert len(test_df) == len(y_true), '테스트 CSV와 예측 결과의 이미지 수가 다릅니다.'
        error_results = []
        for image_idx in range(len(y_true)):
            image_id = str(test_df.iloc[image_idx]['dicom_id'])
            for finding_idx, finding in enumerate(FINDING_NAMES):
                true_label = int(y_true[image_idx, finding_idx])
                if true_label not in BINARY_CLASS_LABELS:
                    continue
                pred_label = int(y_pred_binary[image_idx, finding_idx])
                if true_label != pred_label:
                    error_results.append({'이미지 번호': image_idx + 1, '이미지 ID': image_id, '소견': finding, '실제 정답': BINARY_CLASS_NAMES[true_label], '모델 예측': BINARY_CLASS_NAMES[pred_label], '오류 유형': 'FN' if true_label == 1 else 'FP', '양성 점수': float(y_score[image_idx, finding_idx])})
        error_df = pd.DataFrame(error_results, columns=['이미지 번호', '이미지 ID', '소견', '실제 정답', '모델 예측', '오류 유형', '양성 점수'])
        display(error_df)
        error_counts = error_df.groupby('소견').size().reindex(FINDING_NAMES, fill_value=0).rename('오분류 개수')
        print('===== 소견별 오분류 개수 =====')
        print(error_counts.to_string())
        error_types = error_df.groupby(['소견', '실제 정답', '모델 예측']).size().reset_index(name='개수')
        print('\n===== 소견별 오류 유형 =====')
        print(error_types.to_string(index=False))
        import json
        from datetime import datetime
        SAVE_DIR = PROJECT_DIR / 'evaluation' / 'results' / EXPERIMENT_NAME / MODEL_PATH.parent.name
        SAVE_DIR.mkdir(parents=True, exist_ok=True)

        def save_json(filename, data):
            (SAVE_DIR / filename).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
        assert len(test_df) == len(y_true)
        np.savez_compressed(SAVE_DIR / 'predictions_test.npz', y_true=y_true, y_prob=y_prob, y_pred=y_pred, y_score=y_score, y_pred_binary=y_pred_binary, dicom_id=test_df['dicom_id'].astype(str).to_numpy(dtype=str))
        tables = {'confidence_intervals': ci_df, 'auroc': auroc_df, 'auprc': auprc_df, 'additional_metrics': additional_df, 'finding_groups': group_df, 'misclassifications': error_df}
        for name, table in tables.items():
            table.to_csv(SAVE_DIR / f'{name}.csv', index=False, encoding='utf-8-sig')
        binary_reports = {}
        for i, finding in enumerate(FINDINGS):
            mask = np.isin(y_true[:, i], [0, 1])
            if mask.any():
                binary_reports[finding] = classification_report(y_true[mask, i], y_pred_binary[mask, i], labels=BINARY_CLASS_LABELS, target_names=BINARY_CLASS_NAMES, zero_division=0, output_dict=True)
            else:
                binary_reports[finding] = None
        save_json('classification_reports_binary.json', binary_reports)
        if UNCERTAINTY_STRATEGY == 'u_multiclass':
            multiclass_reports = {finding: classification_report(y_true[:, i], y_pred[:, i], labels=CLASS_LABELS, target_names=CLASS_NAMES, zero_division=0, output_dict=True) for i, finding in enumerate(FINDINGS)}
            save_json('classification_reports_multiclass.json', multiclass_reports)
        positive_score_rule = 'p_positive / (p_negative + p_positive)' if UNCERTAINTY_STRATEGY == 'u_multiclass' else 'softmax(negative_logit, positive_logit)[positive]'
        info = {'evaluated_at': datetime.now().astimezone().isoformat(), 'experiment_name': EXPERIMENT_NAME, 'model_name': MODEL_NAME, 'uncertainty_strategy': UNCERTAINTY_STRATEGY, 'model_path': str(MODEL_PATH), 'test_csv': str(CSV_DIR / 'pilot_test.csv'), 'image_root': str(IMAGE_ROOT), 'test_image_count': len(y_true), 'saved_epoch': checkpoint['epoch'], 'training_config': checkpoint.get('config'), 'input_config': input_config, 'input_config_source': input_config_source, 'checkpoint_input': checkpoint.get('input'), 'finding_order': FINDINGS, 'ground_truth_label_codes': {'0': '음성', '1': '양성', '2': '불확실'}, 'native_class_order': CLASS_NAMES if UNCERTAINTY_STRATEGY == 'u_multiclass' else BINARY_CLASS_NAMES, 'binary_class_order': BINARY_CLASS_NAMES, 'positive_score_rule': positive_score_rule, 'binary_threshold': float(THRESHOLD), 'native_prediction_rule': 'argmax over native classes', 'binary_prediction_rule': 'y_score >= binary_threshold', 'finding_evaluation_rule': '해당 소견의 정답이 0 또는 1인 항목만 평가', 'group_evaluation_rule': '세 소견의 정답이 모두 0 또는 1인 이미지만 평가'}
        info['confidence_intervals'] = ci_config
        save_json('evaluation_info.json', info)
        print('실험 이름:', EXPERIMENT_NAME)
        print('평가 결과 저장 완료:', SAVE_DIR)
        return SAVE_DIR
    finally:
        # 예외가 발생해도 GPU 텐서와 모델 참조 정리
        if model is not None:
            model.to('cpu')
        model = None
        import matplotlib.pyplot as cleanup_plt
        cleanup_plt.close('all')


def run_all_experiments():
    preflight_experiments()
    completed = []
    for exp in EXPERIMENTS:
        try:
            result_dir = evaluate_experiment(exp)
            completed.append(result_dir)
        except Exception:
            print(f"평가 중단: {exp['model_name']} / {exp['uncertainty_strategy']}")
            raise
        finally:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    return completed


## 5. 저장 결과 비교 함수


In [ ]:
def build_comparison(result_dirs):
    rows, group_tables = [], []
    reference_info = reference_ids = reference_true = None
    for result_dir in result_dirs:
        info = json.loads((result_dir / 'evaluation_info.json').read_text(encoding='utf-8'))
        with np.load(result_dir / 'predictions_test.npz', allow_pickle=False) as data:
            ids, truth = data['dicom_id'].copy(), data['y_true'].copy()
        if reference_info is None:
            reference_info, reference_ids, reference_true = info, ids, truth
        else:
            for key in ('finding_order', 'binary_threshold', 'finding_evaluation_rule', 'group_evaluation_rule', 'input_config'):
                if info[key] != reference_info[key]:
                    raise ValueError(f"평가 설정 불일치: {info['experiment_name']} / {key}")
            if not np.array_equal(ids, reference_ids) or not np.array_equal(truth, reference_true):
                raise ValueError('실험 간 테스트 이미지 순서 또는 정답이 다릅니다.')
        auroc = pd.read_csv(result_dir / 'auroc.csv').set_index('소견')
        auprc = pd.read_csv(result_dir / 'auprc.csv').set_index('소견')
        additional = pd.read_csv(result_dir / 'additional_metrics.csv').set_index('소견')
        ci = pd.read_csv(result_dir / 'confidence_intervals.csv')
        reports = json.loads((result_dir / 'classification_reports_binary.json').read_text(encoding='utf-8'))
        for en, ko in zip(FINDINGS, FINDING_NAMES):
            positive = reports[en]['양성'] if reports[en] is not None else {}
            rows.append({
                '실험': info['experiment_name'], '학습 실행 폴더': result_dir.name,
                '소견': ko, '평가 이미지 수': auroc.loc[ko, '평가 이미지 수'],
                '양성 수': auroc.loc[ko, '양성 수'], '공통 AUROC': auroc.loc[ko, '공통 AUROC'],
                '공통 AUPRC (사다리꼴)': auprc.loc[ko, '공통 AUPRC (사다리꼴)'],
                'AP': additional.loc[ko, 'AP'],
                '양성 Precision': positive.get('precision', np.nan),
                '양성 Recall': positive.get('recall', np.nan),
                '양성 F1': positive.get('f1-score', np.nan),
                'Sensitivity': additional.loc[ko, 'Sensitivity'],
                'Specificity': additional.loc[ko, 'Specificity'],
            })
            for metric in CI_METRICS:
                entry = ci[(ci['소견'] == ko) & (ci['지표'] == metric)].iloc[0]
                for column in ['95% CI 하한', '95% CI 상한', '유효 반복 횟수']:
                    rows[-1][f'{metric} {column}'] = entry[column]
        groups = pd.read_csv(result_dir / 'finding_groups.csv')
        groups.insert(0, '실험', info['experiment_name'])
        group_tables.append(groups)
    comparison_df = pd.DataFrame(rows)
    group_comparison_df = pd.concat(group_tables, ignore_index=True)
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    comparison_df.to_csv(RESULTS_ROOT / 'experiment_comparison.csv', index=False, encoding='utf-8-sig')
    group_comparison_df.to_csv(RESULTS_ROOT / 'experiment_group_comparison.csv', index=False, encoding='utf-8-sig')
    display(comparison_df.round(4))
    display(group_comparison_df.round(4))
    print('비교표 저장 완료:', RESULTS_ROOT)
    return comparison_df, group_comparison_df


## 6. 세 모델 자동 평가 및 비교표 생성

이 셀을 실행하면 세 모델의 추론을 순서대로 수행합니다.


In [ ]:
completed_result_dirs = run_all_experiments()
comparison_df, group_comparison_df = build_comparison(completed_result_dirs)